# <font color="#418FDE" size="6.5" uppercase>**Semi Supervised**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Represent labeled and unlabeled samples correctly for semi-supervised estimators. 
- Apply self-training with threshold or k-best pseudo-label selection. 
- Use graph-based label propagation methods and evaluate their assumptions. 


## **1. Unlabeled Data**

### **1.1. Marking Unlabeled Samples**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_01_01.jpg?v=1783851969" width="250">



>* Separate labeled cases from unlabeled ones.
>* Unlabeled data still helps shape training.

>* Unknown labels are not actual classes.
>* Clear marking preserves meaning across datasets.

>* Proper marking prevents corrupted training and evaluation.
>* Unlabeled data still reveals useful structure.



In [ ]:
#@title Python Code - Marking Unlabeled Samples

# This script marks unlabeled targets clearly.
# It uses a tiny semi supervised example.
# Unknown labels differ from real classes.

import numpy as np
import pandas as pd

# Build small feature data.
X = np.array([
    [1.0, 1.2], [1.1, 0.9],
    [3.9, 4.2], [4.1, 3.8],
    [1.2, 1.0], [4.0, 4.1]
])

# Use minus one for unknown labels.
y = np.array([0, 0, 1, 1, -1, -1])

# Validate matching sample counts.
assert X.shape[0] == y.shape[0]
assert set(np.unique(y)).issubset({-1, 0, 1})

# Put data into a table.
df = pd.DataFrame(X, columns=["length_cm", "width_cm"])
df["target"] = y

# Count labeled and unlabeled rows.
unlabeled_mask = df["target"].eq(-1)
labeled_mask = ~unlabeled_mask

# Show the prepared dataset.
print("Dataset preview:")
print(df.to_string(index=False))

# Summarize the target meaning.
print("\nLabeled samples:", int(labeled_mask.sum()))
print("Unlabeled samples:", int(unlabeled_mask.sum()))
print("Known classes:", sorted(df.loc[labeled_mask, "target"].unique().tolist()))

# Contrast wrong and correct meanings.
print("\nImportant idea:")
print("-1 means unknown, not a real class.")



### **1.2. Marking Unlabeled Samples**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_01_02.jpg?v=1783851971" width="250">



>* Unlabeled samples have features but unknown targets.
>* Wrong labels can distort learned boundaries.

>* Unknown labels are not negative labels.
>* Proper marking helps models learn safely.

>* Use one consistent marker for unlabeled data.
>* Consistency prevents errors and supports learning.



In [ ]:
#@title Python Code - Marking Unlabeled Samples

# This script marks unlabeled targets clearly.
# Semi supervised data needs consistent placeholders.
# Unknown labels differ from negative labels.

import numpy as np
import pandas as pd

# Build a tiny feature table.
X = pd.DataFrame({"length_cm": [12, 15, 14, 30, 28, 29]})

# Use minus one for unknown classes.
y = np.array([0, 0, -1, 1, -1, 1])

# Check matching sample counts first.
assert len(X) == len(y), "Features and targets mismatch."

# Create readable status labels.
status = np.where(y == -1, "unlabeled", "labeled")

# Combine features and target markers.
preview = X.assign(target=y, status=status)

# Count labeled and unlabeled rows.
labeled_count = int(np.sum(y != -1))
unlabeled_count = int(np.sum(y == -1))

# Show a small preview.
print(preview.to_string(index=False))

# Explain the key idea briefly.
print()
print(f"Labeled samples: {labeled_count}")
print(f"Unlabeled samples: {unlabeled_count}")
print("Here, -1 means unknown class, not class 0.")



### **1.3. Marking Unlabeled Samples**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_01_03.jpg?v=1783851973" width="250">



>* Separate labeled samples from truly unlabeled ones.
>* Unlabeled data have features, not confirmed targets.

>* Mark unlabeled data explicitly, not ambiguously.
>* Prevents false labels and preserves valid learning.

>* Consistent marking supports teamwork, auditing, and evaluation.
>* Unlabeled data inform, but do not confirm.



In [ ]:
#@title Python Code - Marking Unlabeled Samples

# This example marks unlabeled targets clearly.
# It uses minus one intentionally.
# Features stay complete for every sample.

import numpy as np
import pandas as pd

# Build a tiny classification style dataset.
X = np.array([
    [1.0, 2.0],
    [1.2, 1.8],
    [3.5, 4.0],
    [3.8, 3.6],
])

# Use -1 for unlabeled targets.
y = np.array([0, -1, 1, -1])

# Check matching sample counts first.
assert len(X) == len(y), "Mismatched sample counts."

# Create a small readable table.
df = pd.DataFrame(X, columns=["length_cm", "width_cm"])
df["target"] = y

df["status"] = np.where(df["target"].eq(-1), "unlabeled", "labeled")

# Count labeled and unlabeled rows.
labeled_count = int((y != -1).sum())
unlabeled_count = int((y == -1).sum())

# Show the key idea clearly.
print("Dataset with explicit unlabeled marker:")
print(df.to_string(index=False))
print()
print(f"Labeled samples: {labeled_count}")
print(f"Unlabeled samples: {unlabeled_count}")
print("Meaning: -1 is unknown, not a class.")



## **2. Pseudo Label Selection**

### **2.1. Threshold Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_02_01.jpg?v=1783851964" width="250">



>* Threshold controls which pseudo-labels are accepted.
>* Higher thresholds reduce errors but slow growth.

>* Thresholds depend on confidence reliability and context.
>* Validate thresholds for imbalance and generalization.

>* Start strict, relax thresholds as learning improves.
>* Match thresholds to bias, risk, and data.



In [ ]:
#@title Python Code - Threshold Selection

# Thresholds control cautious pseudo label growth.
# This example compares strict and relaxed choices.
# We use simple probabilities without training.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create tiny unlabeled sample coordinates.
x = np.linspace(-3.0, 3.0, 25)
y = rng.normal(0.0, 0.18, size=x.size)

# Convert positions into class one probabilities.
proba_one = 1.0 / (1.0 + np.exp(-1.7 * x))
pseudo_label = (proba_one >= 0.5).astype(int)

# Measure confidence for threshold selection.
confidence = np.maximum(proba_one, 1.0 - proba_one)
strict_mask = confidence >= 0.90
relaxed_mask = confidence >= 0.70

# Count accepted pseudo labels safely.
strict_count = int(strict_mask.sum())
relaxed_count = int(relaxed_mask.sum())
extra_count = int(relaxed_count - strict_count)

# Print a short teaching summary.
print(f"Total unlabeled samples: {x.size}")
print(f"Accepted at 0.90 threshold: {strict_count}")
print(f"Accepted at 0.70 threshold: {relaxed_count}")
print(f"Extra samples from lower threshold: {extra_count}")

# Build one compact comparison plot.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = np.where(pseudo_label == 1, "tab:orange", "tab:blue")

# Show strict threshold acceptance.
axes[0].scatter(x[~strict_mask], y[~strict_mask], c="lightgray", s=45)
axes[0].scatter(x[strict_mask], y[strict_mask], c=colors[strict_mask], s=55)
axes[0].set_title("Threshold = 0.90")
axes[0].set_xlabel("Feature value")

# Finish strict panel formatting.
axes[0].set_ylabel("Jittered display")
axes[0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0].set_ylim(-0.6, 0.6)
axes[0].grid(alpha=0.25)

# Show relaxed threshold acceptance.
axes[1].scatter(x[~relaxed_mask], y[~relaxed_mask], c="lightgray", s=45)
axes[1].scatter(x[relaxed_mask], y[relaxed_mask], c=colors[relaxed_mask], s=55)
axes[1].set_title("Threshold = 0.70")
axes[1].set_xlabel("Feature value")

# Finish relaxed panel formatting.
axes[1].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_ylim(-0.6, 0.6)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()



### **2.2. K Best Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_02_02.jpg?v=1783851966" width="250">



>* Rank predictions and add only top k.
>* Gradual expansion reduces early labeling mistakes.

>* K controls speed and reliability tradeoff.
>* Per-class selection protects minority categories.

>* Top predictions can still be wrong.
>* Choose k carefully and monitor results.



In [ ]:
#@title Python Code - K Best Selection

# This script demonstrates cautious pseudo label growth.
# We rank unlabeled points by prediction confidence.
# Then we keep only the k best.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two simple labeled clusters.
class0 = rng.normal(loc=[0.0, 0.0], scale=0.45, size=(6, 2))
class1 = rng.normal(loc=[2.2, 2.0], scale=0.45, size=(6, 2))

# Build the labeled training set.
X_labeled = np.vstack((class0, class1))
y_labeled = np.array([0] * 6 + [1] * 6)

# Create unlabeled points near both classes.
X_unlabeled = rng.normal(loc=[1.1, 1.0], scale=1.0, size=(18, 2))

# Check expected array shapes.
assert X_labeled.shape == (12, 2)
assert X_unlabeled.shape == (18, 2)

# Define a tiny nearest centroid model.
def predict_with_confidence(X_train, y_train, X_test):
    classes = np.unique(y_train)
    centroids = np.array([X_train[y_train == c].mean(axis=0) for c in classes])

    # Compute distances to each centroid.
    distances = np.linalg.norm(X_test[:, None] - centroids[None, :], axis=2)
    order = np.argsort(distances, axis=1)

    # Convert distance gap into confidence.
    best = order[:, 0]
    second = order[:, 1]
    gap = distances[np.arange(len(X_test)), second] - distances[np.arange(len(X_test)), best]

    # Normalize confidence safely.
    confidence = gap / (distances[np.arange(len(X_test)), second] + 1e-9)
    labels = classes[best]
    return labels, confidence

# Predict pseudo labels for unlabeled points.
pseudo_labels, confidence = predict_with_confidence(
    X_labeled, y_labeled, X_unlabeled
)

# Choose only the k most confident points.
k = 5
k = min(k, len(X_unlabeled))
selected = np.argsort(confidence)[-k:][::-1]

# Build the expanded training set.
X_added = X_unlabeled[selected]
y_added = pseudo_labels[selected]
X_expanded = np.vstack((X_labeled, X_added))
y_expanded = np.concatenate((y_labeled, y_added))

# Print a short teaching summary.
print("Labeled samples:", len(X_labeled))
print("Unlabeled samples:", len(X_unlabeled))
print("Chosen k value:", k)
print("Selected indices:", selected.tolist())
print("Selected confidences:", np.round(confidence[selected], 3).tolist())
print("Pseudo labels added:", y_added.tolist())
print("Expanded training size:", len(X_expanded))

# Plot labeled, unlabeled, and selected points.
plt.figure(figsize=(6, 5))
plt.scatter(class0[:, 0], class0[:, 1], label="Labeled class 0", s=60)
plt.scatter(class1[:, 0], class1[:, 1], label="Labeled class 1", s=60)
plt.scatter(X_unlabeled[:, 0], X_unlabeled[:, 1], label="Unlabeled pool", alpha=0.35, s=45)
plt.scatter(X_added[:, 0], X_added[:, 1], label="Top k selected", s=140, facecolors="none", edgecolors="black")

# Finish the single teaching plot.
plt.title("K best pseudo label selection")
plt.legend()
plt.tight_layout()
plt.show()



### **2.3. Confidence Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_02_03.jpg?v=1783851967" width="250">



>* Confidence should reflect reliability, not just score.
>* Use trusted pseudo-labels to limit error spread.

>* Confidence needs certainty and clear class separation.
>* Ambiguous predictions make weak pseudo-labels.

>* Use stricter confidence when errors are costly.
>* Monitor class bias and confidence drift.



In [ ]:
#@title Python Code - Confidence Criteria

# This script demonstrates pseudo label confidence choices.
# It compares threshold and top margin selection.
# Small arrays keep the example beginner friendly.

import numpy as np
import pandas as pd

# Set a deterministic random seed.
np.random.seed(7)

# Create tiny prediction probabilities.
probs = np.array([
    [0.92, 0.08],
    [0.51, 0.49],
    [0.80, 0.20],
    [0.60, 0.40],
    [0.55, 0.45],
])

# Name the unlabeled examples.
items = np.array(["A", "B", "C", "D", "E"])

# Validate the probability table.
assert probs.ndim == 2 and probs.shape[1] == 2
assert len(items) == probs.shape[0]

# Compute predicted classes and confidence.
pred_class = probs.argmax(axis=1)
top_conf = probs.max(axis=1)
second_conf = probs.min(axis=1)
margin = top_conf - second_conf

# Apply a simple threshold rule.
threshold = 0.75
keep_threshold = top_conf >= threshold

# Apply a k best margin rule.
k_best = 2
best_margin_idx = np.argsort(margin)[-k_best:][::-1]
keep_kbest = np.zeros(len(items), dtype=bool)
keep_kbest[best_margin_idx] = True

# Build a compact summary table.
summary = pd.DataFrame({
    "item": items,
    "class": pred_class,
    "top_conf": np.round(top_conf, 2),
    "margin": np.round(margin, 2),
    "threshold": keep_threshold,
    "k_best": keep_kbest,
})

# Show the confidence decisions.
print("Confidence criteria for pseudo labels")
print(summary.to_string(index=False))
print()
print("Threshold keeps only very confident examples.")
print("K-best keeps the clearest relative winners.")
print("Item B shows why margin matters.")



## **3. Graph Label Propagation**

### **3.1. Label Similarity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_03_01.jpg?v=1783851975" width="250">



>* Similar samples are likely share labels.
>* Graphs use feature similarity to spread labels.

>* Similarity depends on features and comparison.
>* Good graphs need meaningful sample connections.

>* Labels spread smoothly through similar clusters.
>* Propagation works only with meaningful neighborhoods.



In [ ]:
#@title Python Code - Label Similarity

# Similar points often share nearby labels.
# Graph edges depend on feature similarity.
# Good features create useful propagation neighborhoods.

import numpy as np
import matplotlib.pyplot as plt

# Fix randomness for repeatable teaching output.
rng = np.random.default_rng(7)

# Create two compact sample clusters.
left = rng.normal(loc=[-1.2, 0.0], scale=0.28, size=(18, 2))
right = rng.normal(loc=[1.2, 0.1], scale=0.28, size=(18, 2))

# Stack points into one feature matrix.
X = np.vstack([left, right])
labels = np.array([0] * 18 + [1] * 18)

# Check expected shapes before plotting.
assert X.shape == (36, 2), "Unexpected feature shape."
assert labels.shape == (36,), "Unexpected label shape."

# Choose one point from each cluster.
anchor_left = X[4]
anchor_right = X[26]

# Measure distances from each anchor.
d_left = np.linalg.norm(X - anchor_left, axis=1)
d_right = np.linalg.norm(X - anchor_right, axis=1)

# Convert distances into similarity strengths.
sigma = 0.45
s_left = np.exp(-(d_left ** 2) / (2 * sigma ** 2))
s_right = np.exp(-(d_right ** 2) / (2 * sigma ** 2))

# Summarize how similarity matches cluster labels.
left_same = s_left[labels == 0].mean()
left_other = s_left[labels == 1].mean()
right_same = s_right[labels == 1].mean()
right_other = s_right[labels == 0].mean()

# Print a short interpretation summary.
print("Left anchor same-cluster similarity:", round(left_same, 3))
print("Left anchor other-cluster similarity:", round(left_other, 3))
print("Right anchor same-cluster similarity:", round(right_same, 3))
print("Right anchor other-cluster similarity:", round(right_other, 3))
print("Higher local similarity supports label propagation.")

# Draw points and anchor connections.
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(left[:, 0], left[:, 1], c="tab:blue", label="Class 0")
ax.scatter(right[:, 0], right[:, 1], c="tab:orange", label="Class 1")

# Highlight the two anchor samples.
ax.scatter(*anchor_left, c="navy", s=120, marker="*")
ax.scatter(*anchor_right, c="darkred", s=120, marker="*")

# Connect each anchor to nearest neighbors.
for idx in np.argsort(d_left)[1:5]:
    ax.plot([anchor_left[0], X[idx, 0]], [anchor_left[1], X[idx, 1]], c="navy", alpha=0.45)
for idx in np.argsort(d_right)[1:5]:
    ax.plot([anchor_right[0], X[idx, 0]], [anchor_right[1], X[idx, 1]], c="darkred", alpha=0.45)

# Add a concise teaching title.
ax.set_title("Similar samples form local graph neighborhoods")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()



### **3.2. Label Propagation Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_03_02.jpg?v=1783851977" width="250">



>* Labels spread through similarity graph connections.
>* Useful when few labels, many unlabeled samples.

>* Neighbor labels iteratively shape unlabeled point predictions.
>* Data geometry guides labels in real applications.

>* Transductive method labels only graph’s current samples.
>* Good graphs reveal curved boundaries reliably.



### **3.3. Propagation Assumptions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_03_03.jpg?v=1783851979" width="250">



>* Nearby points should share labels smoothly.
>* Success depends on meaningful feature similarity.

>* Propagation works when clusters mostly share labels.
>* Too many cross-group links can spread errors.

>* Reliable labels and clean graphs are essential.
>* Check similarity, clusters, and anchor labels.



# <font color="#418FDE" size="6.5" uppercase>**Semi Supervised**</font>


In this lecture, you learned to:
- Represent labeled and unlabeled samples correctly for semi-supervised estimators. 
- Apply self-training with threshold or k-best pseudo-label selection. 
- Use graph-based label propagation methods and evaluate their assumptions. 

In the next Module (Module 20), we will go over 'None'